In [ ]:
# ─── Célula 1: Detecção de ambiente ───────────────────────────────────────────
import os, sys

try:
    import google.colab
    COLAB = True
    print('Ambiente: Google Colab')
except ImportError:
    COLAB = False
    print('Ambiente: Local')

import torch
GPU = torch.cuda.is_available()
print(f'GPU disponível: {GPU}')
if GPU:
    print(f'  {torch.cuda.get_device_name(0)} — VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ─── Célula 2: Instalação de dependências (só Colab) ──────────────────────────
if COLAB:
    %pip install -q transformers datasets seqeval torch accelerate

In [ ]:
# ─── Célula 3: Caminhos ────────────────────────────────────────────────────────
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE      = '/content/drive/MyDrive/Mestrado/Dissertação/AnonClin/Experimentos/Experimento 001'
    DRIVE_DATA_PATH = os.path.join(DRIVE_BASE, 'dados')
    OUTPUT_BASE     = '/content/modelos'   # local Colab (~78GB livres) — evita lotar o Drive

    TRAIN_PATH = os.path.join(DRIVE_DATA_PATH, 'train.conll')
    DEV_PATH   = os.path.join(DRIVE_DATA_PATH, 'dev.conll')
    TEST_PATH  = os.path.join(DRIVE_DATA_PATH, 'test.conll')

    REPO_URL = 'https://github.com/glauciohenriquesilva/anonimizacao-textos-clinicos-ptbr.git'
    REPO_DIR = '/content/anonclin'
    import subprocess, shutil

    # Re-clona se diretório não existir ou estiver incompleto
    if not os.path.exists(os.path.join(REPO_DIR, 'ner')):
        if os.path.exists(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR],
                                capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f'Clone falhou:\n{result.stderr}')
        print('Clone OK')
    else:
        print('Repo já clonado')

    # Garante __init__.py nos pacotes Python (necessário para importar sem Django)
    for pkg in [REPO_DIR, f'{REPO_DIR}/ner', f'{REPO_DIR}/ner/services']:
        init = os.path.join(pkg, '__init__.py')
        if not os.path.exists(init):
            open(init, 'w').close()

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

else:
    # Local: notebook está em notebooks/, repositório está um nível acima
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)

    TRAIN_PATH  = os.path.join(REPO_ROOT, 'outputs', 'ner', 'train.conll')
    DEV_PATH    = os.path.join(REPO_ROOT, 'outputs', 'ner', 'dev.conll')
    TEST_PATH   = os.path.join(REPO_ROOT, 'outputs', 'ner', 'test.conll')
    OUTPUT_BASE = os.path.join(REPO_ROOT, 'outputs', 'ner', 'modelos')

os.makedirs(OUTPUT_BASE, exist_ok=True)
print(f'train : {TRAIN_PATH} — existe: {os.path.exists(TRAIN_PATH)}')
print(f'dev   : {DEV_PATH}   — existe: {os.path.exists(DEV_PATH)}')
print(f'test  : {TEST_PATH}  — existe: {os.path.exists(TEST_PATH)}')
print(f'output: {OUTPUT_BASE}')

In [ ]:
# ─── Célula 4: Configuração ────────────────────────────────────────────────────
# !! ALTERE AQUI PARA CADA MODELO !!

MODEL_ID = 'pucpr/biobertpt-clin'                          # 2.3.2
# MODEL_ID = 'pierreguillou/bert-base-cased-pt-lenerbr'    # 2.3.3
# MODEL_ID = 'jhu-clsp/mmBERT'                             # 2.3.4 — requer A100
# MODEL_ID = 'answerdotai/ModernBERT-base'                 # 2.3.5 — requer A100

EPOCHS     = 5
BATCH_SIZE = 16   # reduzir para 8 se aparecer OOM

MODEL_SLUG = MODEL_ID.split('/')[-1]   # ex: biobertpt-clin
OUTPUT_DIR = os.path.join(OUTPUT_BASE, MODEL_SLUG)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Modelo  : {MODEL_ID}')
print(f'Epochs  : {EPOCHS}')
print(f'Batch   : {BATCH_SIZE}')
print(f'Saída   : {OUTPUT_DIR}')

In [ ]:
# ─── Célula 5: Treinamento ─────────────────────────────────────────────────────
# bert.py não tem dependência do Django — importa direto
from ner.services.bert import treinar_bert

trainer, tokenizer, label2id, id2label = treinar_bert(
    model_id      = MODEL_ID,
    caminho_train = TRAIN_PATH,
    caminho_dev   = DEV_PATH,
    caminho_saida = OUTPUT_DIR,
    epochs        = EPOCHS,
    batch_size    = BATCH_SIZE,
)

print(f'\nModelo salvo em: {OUTPUT_DIR}')

In [ ]:
# ─── Célula 6: Avaliação no conjunto de teste ──────────────────────────────────
from ner.services.bert import carregar_conll
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline
from seqeval.metrics import classification_report, f1_score
import torch

tokenizer_eval = AutoTokenizer.from_pretrained(OUTPUT_DIR)
modelo_eval    = AutoModelForTokenClassification.from_pretrained(OUTPUT_DIR)
modelo_eval.eval()
device = 0 if torch.cuda.is_available() else -1

pipe = pipeline(
    'token-classification',
    model=modelo_eval,
    tokenizer=tokenizer_eval,
    aggregation_strategy='simple',
    device=device,
)

teste  = carregar_conll(TEST_PATH)
y_true, y_pred = [], []

MAX_WORD_TOKENS = 500  # BERT suporta 512 subtokens; 500 word-tokens é margem segura

for ex in teste:
    tokens_orig = ex['tokens']
    tokens_eval = tokens_orig[:MAX_WORD_TOKENS]  # trunca antes de passar à pipeline
    texto = ' '.join(tokens_eval)
    resultado = pipe(texto)

    pred_labels = ['O'] * len(tokens_orig)  # tamanho original (excedente fica 'O')
    pos_acum = 0
    tok_starts = []
    for tok in tokens_eval:
        tok_starts.append(pos_acum)
        pos_acum += len(tok) + 1  # +1 pelo espaço

    for ent in resultado:
        for i, start in enumerate(tok_starts):
            if start >= ent['start'] and start < ent['end']:
                prefix = 'B' if start == ent['start'] else 'I'
                pred_labels[i] = f"{prefix}-{ent['entity_group']}"

    y_true.append(ex['labels'])
    y_pred.append(pred_labels)

f1 = f1_score(y_true, y_pred)
print(f'F1 Entity-level (micro): {f1:.4f}')
print()
print(classification_report(y_true, y_pred))

In [ ]:
# ─── Célula 7: Empacotar modelo para importar no AnonClin ─────────────────────
import shutil, glob

# Remove checkpoints intermediários — mantém só o modelo final (~700MB vs 5GB+)
for ckpt in glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*')):
    shutil.rmtree(ckpt)
print('Checkpoints removidos.')

# Zip sempre em /content/ para não usar cota do Drive
zip_dir = '/content/modelos' if COLAB else OUTPUT_BASE
os.makedirs(zip_dir, exist_ok=True)
zip_path = os.path.join(zip_dir, f'bert_model_{MODEL_SLUG}')
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f'Arquivo gerado: {zip_path}.zip')

if COLAB:
    from google.colab import files
    files.download(f'{zip_path}.zip')
    print('Download iniciado automaticamente.')
else:
    print(f'Arquivo disponível em: {zip_path}.zip')
    print('Para importar no AnonClin: NER → Avaliação → upload do .zip')